# NSCLC scRNA-seq: NSCLC_7pt (P1–P4) + public GSE148071: merge, preprocess, and test for batch effects before integration (Seurat / R)

Same preprocessing as the stage-1 tutorial (`analysis_stage1.ipynb`), applied to two NSCLC datasets:

1. Load the four NSCLC_7pt samples (10x; RNA, TotalSeq antibodies, CellPlex tags), **merge** them and save the object
2. Load four patients of the public dataset **GSE148071** (another study, another platform), keep the genes both datasets share, and merge them in. **The dataset label (`NSCLC_7pt` / `GSE148071`) is the batch key.**
3. **Ambient RNA check (SoupX: contamination estimated, counts not corrected) → doublets (scDblFinder)**, both on the unfiltered cells and before any QC metric → QC metrics → per-sample MAD thresholds → filtering
4. Log-normalization → 2,000 HVGs → scaling → PCA
5. **Is the structure driven by dataset / sample (batch) or by biology?** PCA-based diagnostics only. **No neighbour graph, clustering or UMAP is built here.**

> **Findings so far (run with NSCLC_7pt P1–P4 only)**
> * **No technical batch effect between P1–P4.** Cell compartment explains ~39% of the variance in the leading PCs, sample ~9%; within the shared immune and stromal compartments sample adds only ~1%. Silhouette width by sample ≈ 0 (−0.02), by compartment 0.23.
> * **Why:** P1–P4 were **multiplexed with CellPlex**: pooled before loading, so they shared one 10x capture, library prep and sequencing run.
> * **What differs between P1–P4 is biology:** the malignant / epithelial cells and the cell-type composition of each tumour.
>
> **What adding GSE148071 tests (re-run the notebook to get the numbers)**
> * GSE148071 differs from NSCLC_7pt in everything that creates a **technical batch effect**: lab, tissue handling (biopsies), platform (Singleron microwells, not 10x Chromium), chemistry, gene annotation, sequencing depth and the authors' own cell filtering.
> * A batch effect shows up as the **same cell type** (e.g. T/NK cells) splitting by dataset in the PCA: `dataset` explains PC variance *within* shared compartments, and the silhouette by dataset is > 0. That is where integration (e.g. Harmony with `dataset` as the key) becomes necessary.
> * P1–P4 are the **negative control**: they should still mix within NSCLC_7pt.

**All paths are relative to the project root** (the folder with `install.sh`; this notebook finds it itself):

| | Location |
|---|---|
| Input: NSCLC_7pt | `NSCLC_7pt/P1/` … `P4/` (`barcodes.tsv.gz`, `features.tsv.gz`, `matrix.mtx.gz`) |
| Input: GSE148071 | `data/GSE148071/GSM*_P*_exp.txt` (genes × barcodes text matrices) |
| Output: merged raw NSCLC_7pt object | `data/nsclc7pt_merged.rds` |
| Output: merged raw NSCLC_7pt + GSE148071 object | `data/nsclc_combined_merged.rds` |
| Output: preprocessed object (pre-integration) | `data/nsclc_combined_preInt.rds` |
| Output: figures / tables | `data/figures/NSCLC_combined/`, `data/tables/NSCLC_combined/` |

`data/` and the inputs are git-ignored. **Requirements:** `bash install.sh` from the project root (conda env `seurat_env`).

### About the datasets

**NSCLC_7pt (10x, multiplexed)**

The 7 NSCLC donors were pooled with 10x CellPlex and processed with Cell Ranger 6.1.2 `multi`. Each folder P1–P4 is one donor's per-sample output (`sample_filtered_feature_bc_matrix`). The dominant CellPlex tag of the cells is CMO301 in P1, CMO302 in P2, CMO303 in P3 and CMO304 in P4. All four folders share the same feature list:

| Feature type | Features | Used for |
|---|---|---|
| Gene Expression | 36,601 genes | `RNA` assay: all of the analysis below |
| Antibody Capture | 9 TotalSeq antibodies: CD3, CD4, CD8, CD11c, CD14, CD16, CD19, CD56, CD45 | `ADT` assay: kept in the object for later annotation |
| Multiplexing Capture | 12 CellPlex tags: CMO301 to CMO312 | `CMO` assay: confirms which donor each folder holds |

The folder name is the sample label (`sample` = P1 to P4). Within NSCLC_7pt it is the negative-control batch label in Section H.

**Why the design matters for batch effects.** All four donors went through the *same* capture, reverse transcription, library prep and sequencing. Technical differences between samples should therefore be minimal. Differences between P1–P4 are expected to be mostly donor / tumour biology.

**GSE148071 (public)**

Wu *et al.*, *Nat Commun* 2021: biopsies (needle, bronchoscopy or lymph node) from 42 patients with advanced NSCLC. GEO holds only processed count matrices; the submitter is Singleron, so the libraries come from Singleron's microwell platform rather than 10x Chromium (also visible in the 12-nt cell barcodes). Four patients are used here:

| File in `data/GSE148071/` | Cells | Libraries (barcode prefix) | `sample` |
|---|---|---|---|
| `GSM4453576_P1_exp.txt` | 5,317 | 1, 2 | GSE148071_P1 |
| `GSM4453578_P3_exp.txt` | 7,835 | 4, 5 | GSE148071_P3 |
| `GSM4453582_P7_exp.txt` | 3,059 | 12, 13 | GSE148071_P7 |
| `GSM4453585_P10_exp.txt` | 3,740 | 17 | GSE148071_P10 |

* Each file is a dense, tab-separated **genes × barcodes** matrix of integer UMI counts. The header row holds only the barcodes; the first column of every other row is the gene symbol.
* The cells were **already filtered by the authors** (≥ ~200 and ≤ ~5,000 genes per cell), so the QC below mostly re-checks their choices.
* **Different gene annotation:** 29,527 genes vs 36,601 in the Cell Ranger reference; **23,819 are shared and only these are kept**. A gene missing from one dataset would otherwise read as 0 in all its cells, an artefact that looks exactly like a batch effect. This also drops MT-RNR1/2 and the mitochondrial tRNAs (present only in GSE148071), so %MT is measured on the same 13 protein-coding mitochondrial genes in both datasets. All markers used in Section H are shared.

**Labels used below**

| Column | Values | Role |
|---|---|---|
| `dataset` | NSCLC_7pt, GSE148071 | **batch key** (study / platform) |
| `sample` | P1–P4, GSE148071_P1 … GSE148071_P10 | patient; GSE148071 labels are prefixed so patient numbers do not collide |
| `library` | NSCLC_7pt: the sample; GSE148071: sample + barcode prefix | unit for doublet detection |
| `soup_channel` | NSCLC_7pt (one pooled capture); GSE148071: each library | unit for the ambient-RNA check (SoupX) |

## Setup

In [ ]:
# WSL2: `bash install.sh` installs everything (conda env seurat_env, kernel "R 4.4.3 (seurat_env)").
# Windows R: run once if the packages are missing:
# install.packages(c("Seurat", "ggplot2", "patchwork"))

suppressPackageStartupMessages({
  library(Seurat)
  library(Matrix)
  library(ggplot2)
  library(patchwork)
})

seed <- 198716                       # same seed as the PBMC stage-1 notebook
set.seed(seed)
options(repr.plot.width = 10, repr.plot.height = 6, repr.plot.res = 120)  # inline plot size in Jupyter
# ---- Runtime per stage ----
# mark_stage("name") is called at the start of each stage; the "Runtime per stage" block at the end turns the marks
# into a table. Wall-clock (elapsed) times are meaningful for an uninterrupted top-to-bottom run (Run All / nbconvert);
# CPU time excludes the time the kernel sat idle between cells (it can exceed elapsed time with a multithreaded BLAS).
stage_marks <- list()
mark_stage <- function(stage) {
  pt <- proc.time()
  stage_marks[[stage]] <<- list(time = as.numeric(Sys.time()), cpu = unname(pt[["user.self"]] + pt[["sys.self"]]))
  invisible(stage)
}
mark_stage("Setup")

packageVersion("Seurat")

In [ ]:
# ---- All paths are relative to the project root: the folder that holds install.sh and scripts/ ----
# Jupyter starts in the notebook's own folder (scripts/seurat_analysis/), so look upwards from there for the root.
find_project_root <- function(start = getwd()) {
  dir <- normalizePath(start, winslash = "/", mustWork = TRUE)
  repeat {
    if (file.exists(file.path(dir, "install.sh")) && dir.exists(file.path(dir, "scripts"))) return(dir)
    parent <- dirname(dir)
    if (parent == dir) stop("Project root (folder with install.sh and scripts/) not found above ", start)
    dir <- parent
  }
}
base_dir    <- find_project_root()
cat("Project root:", base_dir, "\n")
data_root   <- file.path(base_dir, "NSCLC_7pt")           # 10x input: one folder per sample (git-ignored)
gse_dir     <- file.path(base_dir, "data", "GSE148071")   # public GSE148071 text matrices (inside data/, git-ignored)
results_dir <- file.path(base_dir, "data")                # every result goes here; data/ is git-ignored, never pushed to GitHub
fig_dir     <- file.path(results_dir, "figures", "NSCLC_combined")
table_dir   <- file.path(results_dir, "tables",  "NSCLC_combined")
merged_rds   <- file.path(results_dir, "nsclc7pt_merged.rds")        # NSCLC_7pt only, merged raw object (before QC)
combined_rds <- file.path(results_dir, "nsclc_combined_merged.rds")  # NSCLC_7pt + GSE148071, merged raw object (before QC)
rds_file     <- file.path(results_dir, "nsclc_combined_preInt.rds")  # processed object (before Harmony)

# One 10x folder per sample; the folder name becomes the sample label. Add more names here (e.g. "P5").
sample_names <- c("P1", "P2", "P3", "P4")
sample_dirs  <- setNames(file.path(data_root, sample_names), sample_names)

required_files <- c("barcodes.tsv.gz", "features.tsv.gz", "matrix.mtx.gz")
for (s in names(sample_dirs)) {
  missing_files <- required_files[!file.exists(file.path(sample_dirs[[s]], required_files))]
  if (length(missing_files) > 0) {
    stop("Sample ", s, ": missing ", paste(missing_files, collapse = ", "), " in ", sample_dirs[[s]])
  }
}

# GSE148071: one genes x barcodes text matrix per patient (GSM<id>_P<n>_exp.txt); the label is "GSE148071_P<n>"
gse_id    <- "GSE148071"
gse_files <- list.files(gse_dir, pattern = "^GSM[0-9]+_.+_exp\\.txt(\\.gz)?$", full.names = TRUE)
if (length(gse_files) == 0) stop("No GSM*_exp.txt files in ", gse_dir)
names(gse_files) <- paste0(gse_id, "_", sub("^GSM[0-9]+_(.+)_exp\\.txt(\\.gz)?$", "\\1", basename(gse_files)))
gse_files <- gse_files[order(nchar(names(gse_files)), names(gse_files))]   # P1, P3, P7, P10

dir.create(fig_dir,   showWarnings = FALSE, recursive = TRUE)
dir.create(table_dir, showWarnings = FALSE, recursive = TRUE)

# Show a plot inline and save a copy in fig_dir
save_plot <- function(p, filename, width = 10, height = 6) {
  ggsave(file.path(fig_dir, filename), plot = p, width = width, height = height, dpi = 300, bg = "white")
  p
}

# Fraction of total variance explained by each PC (Scanpy: adata.uns['pca']['variance_ratio'])
pca_variance_ratio <- function(obj, reduction = "pca") {
  stdev <- Stdev(obj, reduction = reduction)
  total <- obj[[reduction]]@misc$total.variance
  data.frame(PC = seq_along(stdev), variance_ratio = stdev^2 / total)
}

sample_dirs
gse_files

#### A: Load the samples and merge them

Each folder becomes its own Seurat object with three assays (`RNA`, `ADT`, `CMO`), and the four objects are merged. `merge()` prefixes every cell name with its sample (`P1_AAACCCAG…-1`). In Seurat 5 it also keeps each sample's counts in a separate layer (`counts.P1`, `counts.P2`, …), so the layers are joined back into one `counts` layer per assay.

In [ ]:
mark_stage("A: Load and merge NSCLC_7pt")

# All samples must share the same reference genes and antibody panel
read_features <- function(sample_dir) {
  read.delim(file.path(sample_dir, "features.tsv.gz"), header = FALSE,
             col.names = c("gene_ids", "gene_symbol", "feature_types"))
}
feature_tables <- lapply(sample_dirs, read_features)
same_features  <- vapply(feature_tables, identical, logical(1), feature_tables[[1]])
if (!all(same_features)) stop("features.tsv.gz differs for: ", paste(names(which(!same_features)), collapse = ", "))
table(feature_tables[[1]]$feature_types)

In [ ]:
# Scanpy: sc.read_10x_mtx(path, var_names='gene_symbols', gex_only=False)
# Read10X returns one matrix per feature type: "Gene Expression", "Antibody Capture", "Multiplexing Capture"
read_10x_sample <- function(sample_dir, sample_name) {
  mats     <- Read10X(data.dir = sample_dir, gene.column = 2)            # gene symbols as row names
  if (!is.list(mats)) mats <- list("Gene Expression" = mats)
  features <- read_features(sample_dir)

  obj <- CreateSeuratObject(counts = mats[["Gene Expression"]], project = sample_name,
                            min.cells = 0, min.features = 0)
  obj$sample <- sample_name

  # TotalSeq antibodies. Read10X makes names unique across all feature types, so antibodies named like a gene
  # (CD4, CD14, CD19) come back as "CD4.1" etc.; restore the panel names (ADT is its own assay).
  if (!is.null(mats[["Antibody Capture"]])) {
    adt <- mats[["Antibody Capture"]]
    rownames(adt) <- features$gene_symbol[features$feature_types == "Antibody Capture"]
    obj[["ADT"]] <- CreateAssay5Object(counts = adt)
  }

  # CellPlex tags: dominant tag per cell, and its share of the cell's tag counts (how clean the call is)
  if (!is.null(mats[["Multiplexing Capture"]])) {
    cmo <- mats[["Multiplexing Capture"]]
    obj[["CMO"]] <- CreateAssay5Object(counts = cmo)
    cmo_mat <- t(as.matrix(cmo))                                         # cells x tags (small)
    obj$cmo_tag      <- colnames(cmo_mat)[max.col(cmo_mat, ties.method = "first")]
    obj$cmo_top_frac <- apply(cmo_mat, 1, max) / pmax(rowSums(cmo_mat), 1)
  }
  obj
}

sample_objs <- Map(read_10x_sample, sample_dirs, names(sample_dirs))
sample_objs

In [ ]:
# Merge all samples into one object; cell names get the sample as prefix (P1_<barcode>, ...)
if (length(sample_objs) < 2) stop("merge() needs at least two samples")
nsclc_merged <- merge(x = sample_objs[[1]], y = sample_objs[-1],
                      add.cell.ids = names(sample_objs), project = "NSCLC_7pt")
rm(sample_objs)

Layers(nsclc_merged[["RNA"]])     # one counts layer per sample: counts.P1, counts.P2, ...

# Join the per-sample layers into a single "counts" layer in every assay
for (assay in Assays(nsclc_merged)) {
  nsclc_merged <- JoinLayers(nsclc_merged, assay = assay)
}
DefaultAssay(nsclc_merged) <- "RNA"
nsclc_merged$sample <- factor(nsclc_merged$sample, levels = names(sample_dirs))

Layers(nsclc_merged[["RNA"]])
nsclc_merged

In [ ]:
cat("Number of cells =", ncol(nsclc_merged), "\n")
cat("Number of genes =", nrow(nsclc_merged), "\n")

# Scanpy: var_names_make_unique() / obs_names_make_unique()
# Read10X() already makes duplicated gene symbols unique (e.g. "GENE", "GENE.1"); confirm nothing is duplicated
c(duplicated_genes = sum(duplicated(rownames(nsclc_merged))),
  duplicated_cells = sum(duplicated(colnames(nsclc_merged))))

##### Samples

Each sample should hold one donor, so nearly all cells in a sample should share one dominant CellPlex tag. Every tag in use shows a large total in its own sample; the others only carry ambient counts, a few orders of magnitude lower.

In [ ]:
table(sample = nsclc_merged$sample)
table(sample = nsclc_merged$sample, cmo_tag = nsclc_merged$cmo_tag)
tapply(nsclc_merged$cmo_top_frac, nsclc_merged$sample, summary)

In [ ]:
# Total counts per CellPlex tag in each sample
cmo_counts <- LayerData(nsclc_merged, assay = "CMO", layer = "counts")
cmo_totals <- do.call(rbind, lapply(levels(nsclc_merged$sample), function(s) {
  cells_s <- colnames(nsclc_merged)[nsclc_merged$sample == s]
  data.frame(sample = s, tag = rownames(cmo_counts),
             total_counts = Matrix::rowSums(cmo_counts[, cells_s, drop = FALSE]))
}))
rm(cmo_counts)

options(repr.plot.width = 12, repr.plot.height = 6)
p <- ggplot(cmo_totals, aes(x = tag, y = total_counts + 1)) +
  geom_col(fill = "steelblue") +
  scale_y_log10() +
  facet_wrap(~ sample, ncol = 2) +
  labs(title = "Total counts per CellPlex tag, by sample", x = NULL, y = "Total counts + 1 (log scale)") +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))
save_plot(p, "01_cmo_tag_totals_by_sample.png", width = 12, height = 6)

In [ ]:
# Save the merged raw object (all genes and cells, before QC)
saveRDS(nsclc_merged, file = merged_rds)

#### A2: Add GSE148071 (dataset label = batch key)

The NSCLC_7pt object above is saved as it is (`data/nsclc7pt_merged.rds`). The four GSE148071 matrices are read, converted to sparse matrices, restricted to the genes shared with NSCLC_7pt, and merged in. From here on, `nsclc_merged` holds both datasets. The ADT and CMO assays exist only for NSCLC_7pt cells (GSE148071 has no antibody or CellPlex data); Seurat 5 allows an assay to cover a subset of cells.

In [ ]:
mark_stage("A2: Load GSE148071 and merge")

# Read one GSE148071 matrix (genes x barcodes, tab-separated, integer UMI counts).
# The header holds only the barcodes (one field fewer than the data rows), so it is read on its own.
# data.table::fread (installed with Seurat) is much faster than read.table on these 0.2-0.5 GB files.
read_gse_matrix <- function(path, block_size = 2000) {
  barcodes <- strsplit(readLines(path, n = 1), "\t", fixed = TRUE)[[1]]
  barcodes <- barcodes[nzchar(barcodes)]
  df <- data.table::fread(path, skip = 1, header = FALSE, sep = "\t", showProgress = FALSE, data.table = FALSE)
  if (ncol(df) != length(barcodes) + 1) {
    stop(basename(path), ": ", ncol(df) - 1, " value columns but ", length(barcodes), " barcodes in the header")
  }
  genes <- df[[1]]
  # Dense -> sparse in blocks of columns, so the full dense matrix is never held twice in memory
  blocks <- split(seq_along(barcodes), ceiling(seq_along(barcodes) / block_size))
  mat <- do.call(cbind, unname(lapply(blocks, function(j) as(as.matrix(df[, j + 1, drop = FALSE]), "CsparseMatrix"))))
  rm(df)
  if (any(mat@x != round(mat@x))) warning(basename(path), ": non-integer values; these may not be raw UMI counts")
  # Same feature-name rules as Read10X / Seurat: unique symbols, "_" replaced by "-"
  dimnames(mat) <- list(make.unique(gsub("_", "-", genes)), barcodes)
  mat
}

gse_counts <- lapply(gse_files, read_gse_matrix)
gse_dims <- t(sapply(gse_counts, dim))
colnames(gse_dims) <- c("genes", "cells")
gse_dims

# Barcode prefixes ("1_", "2_", ...) mark the sequencing libraries within each patient
lapply(gse_counts, function(m) table(library = sub("_.*", "", colnames(m))))

In [ ]:
# Keep only the genes present in both datasets (in NSCLC_7pt order). A gene missing from one dataset
# would read as 0 in all of its cells: a pure artefact that would look like a batch effect.
genes_7pt    <- rownames(nsclc_merged[["RNA"]])
genes_gse    <- Reduce(intersect, lapply(gse_counts, rownames))
shared_genes <- genes_7pt[genes_7pt %in% genes_gse]
c(NSCLC_7pt = length(genes_7pt), GSE148071 = length(genes_gse), shared = length(shared_genes))
grep("^MT-", shared_genes, value = TRUE)          # %MT is computed on these genes in both datasets

# NSCLC_7pt: label the dataset and the library (the CellPlex sample), and restrict RNA to the shared genes.
# subset(object, features = ...) would also cut the ADT / CMO assays, so only the RNA assay is replaced.
nsclc_merged$dataset <- "NSCLC_7pt"
nsclc_merged$library <- as.character(nsclc_merged$sample)
suppressWarnings(nsclc_merged[["RNA"]] <- subset(nsclc_merged[["RNA"]], features = shared_genes))

# GSE148071: one Seurat object per patient; cell names follow merge()'s style: <sample>_<barcode>
make_gse_object <- function(counts, sample_name) {
  counts <- counts[shared_genes, , drop = FALSE]
  barcodes <- colnames(counts)
  colnames(counts) <- paste0(sample_name, "_", barcodes)
  obj <- CreateSeuratObject(counts = counts, project = sample_name, min.cells = 0, min.features = 0)
  obj$sample  <- sample_name
  obj$dataset <- gse_id
  obj$library <- paste0(sample_name, "_lib", sub("_.*", "", barcodes))
  obj
}
gse_objs <- Map(make_gse_object, gse_counts, names(gse_counts))
rm(gse_counts)

nsclc_merged <- merge(x = nsclc_merged, y = unname(gse_objs), project = "NSCLC_combined")
rm(gse_objs)
for (assay in Assays(nsclc_merged)) {
  nsclc_merged <- JoinLayers(nsclc_merged, assay = assay)
}
DefaultAssay(nsclc_merged) <- "RNA"

nsclc_merged$dataset <- factor(nsclc_merged$dataset, levels = c("NSCLC_7pt", gse_id))
nsclc_merged$sample  <- factor(as.character(nsclc_merged$sample), levels = c(names(sample_dirs), names(gse_files)))
nsclc_merged$library <- factor(nsclc_merged$library)

# UMIs and genes per cell were counted on all NSCLC_7pt genes; recount them on the shared genes for every cell
rna_counts <- LayerData(nsclc_merged, assay = "RNA", layer = "counts")
nsclc_merged$nCount_RNA   <- Matrix::colSums(rna_counts)
nsclc_merged$nFeature_RNA <- Matrix::colSums(rna_counts > 0)
rm(rna_counts)

nsclc_merged
sapply(Assays(nsclc_merged), function(a) ncol(nsclc_merged[[a]]))   # ADT / CMO: NSCLC_7pt cells only
table(sample = nsclc_merged$sample, dataset = nsclc_merged$dataset)
c(duplicated_cells = sum(duplicated(colnames(nsclc_merged))))

# Save the merged raw object of both datasets (shared genes, all cells, before QC)
saveRDS(nsclc_merged, file = combined_rds)

#### B: Ambient RNA check (SoupX): contamination estimated, counts not corrected

Free mRNA from lysed cells floats in the cell suspension and is captured in every droplet / microwell together with the cell's own mRNA ("soup"). It adds counts of highly abundant genes from other cell types, e.g. immunoglobulins from plasma cells or surfactant genes from alveolar cells, to **every** cell. Here SoupX only **estimates the contamination fraction ρ** for each capture channel. **The counts are not corrected**: all later sections use the uncorrected counts.

* **Channels:** NSCLC_7pt P1–P4 were pooled into one 10x capture, so they share one soup. Each GSE148071 library (barcode prefix) is its own channel.
* **No empty droplets available:** only filtered cell matrices exist for both datasets (no Cell Ranger `raw_feature_bc_matrix`). The soup profile is approximated by the pooled expression of all cells in the channel.
* **Estimating ρ without clustering:** genes that are very high in a few cell types and absent from the rest (haemoglobins, immunoglobulins, surfactants) are used as "non-expressed" sets; in cells that do not express a set, its counts can only come from the soup. Every cell is its own group (`clusters = FALSE`). If ρ cannot be estimated for a channel, 5% is reported and flagged.
* **Why no correction here:** without clusters, `SoupX::adjustCounts` builds a dense genes × cells matrix one cell at a time, which takes too long for this demonstration. Different soups (e.g. more surfactant RNA in one study) can add to the differences between datasets, so keep ρ in mind when reading Section H.

**Full analysis: include the correction** before doublets and QC (Sections C–D), then run the rest of the notebook on the corrected counts. Per capture channel:

```r
# 1) Soup profile from the empty droplets of the capture, if available
#    (NSCLC_7pt, Cell Ranger multi: outs/multi/count/raw_feature_bc_matrix, shared by all CellPlex samples)
tod <- Read10X("path/to/outs/multi/count/raw_feature_bc_matrix")[["Gene Expression"]]
toc <- LayerData(nsclc_merged, assay = "RNA", layer = "counts")[, nsclc_merged$soup_channel == "NSCLC_7pt"]
sc  <- SoupX::SoupChannel(tod = tod[rownames(toc), ], toc = toc)
#    No raw matrix (GSE148071): use the pooled-cell soup profile, as in the cell below

# 2) Clusters from a first-pass clustering make the estimate robust and adjustCounts fast
sc        <- SoupX::setClusters(sc, setNames(as.character(first_pass_clusters[colnames(toc)]), colnames(toc)))
sc        <- SoupX::autoEstCont(sc)                       # or calculateContaminationFraction() as below
corrected <- SoupX::adjustCounts(sc, roundToInt = TRUE)

# 3) After all channels are corrected (corrected_all = cbind of the channels, in the object's cell order):
#    keep the raw counts, replace the RNA counts, recount the library sizes, then continue with Section C
nsclc_merged[["RNA_raw"]] <- CreateAssay5Object(counts = LayerData(nsclc_merged, assay = "RNA", layer = "counts"))
LayerData(nsclc_merged, assay = "RNA", layer = "counts") <- corrected_all
nsclc_merged$nCount_RNA   <- Matrix::colSums(corrected_all)
nsclc_merged$nFeature_RNA <- Matrix::colSums(corrected_all > 0)
```

In [ ]:
mark_stage("B: Ambient RNA check (SoupX)")

# One SoupX channel per capture: NSCLC_7pt (one CellPlex pool) and each GSE148071 library
nsclc_merged$soup_channel <- factor(ifelse(nsclc_merged$dataset == "NSCLC_7pt", "NSCLC_7pt",
                                           as.character(nsclc_merged$library)))

# Genes that are abundant in a few cell types and absent from the others: their counts in other cells = soup
soup_gene_sets <- list(
  haemoglobin    = c("HBB", "HBA1", "HBA2"),
  immunoglobulin = c("IGKC", "IGHG1", "IGHG3", "IGHG4", "IGHA1", "IGHM", "IGLC2", "IGLC3", "JCHAIN"),
  surfactant     = c("SFTPC", "SFTPB", "SFTPA1", "SFTPA2")
)
soup_gene_sets <- lapply(soup_gene_sets, intersect, rownames(nsclc_merged))
soup_gene_sets <- soup_gene_sets[lengths(soup_gene_sets) > 0]
rho_fallback   <- 0.05                      # reported (and flagged) only if rho cannot be estimated for a channel

# Contamination fraction of one channel. Only the estimate: SoupX::adjustCounts() is NOT run (see the note above)
estimate_soup <- function(toc, channel) {
  # No empty droplets: tod = toc, and the soup profile is the pooled expression of all cells in the channel
  sc <- SoupX::SoupChannel(tod = toc, toc = toc, calcSoupProfile = FALSE)
  sc <- SoupX::setSoupProfile(sc, data.frame(est    = Matrix::rowSums(toc) / sum(toc),
                                             counts = Matrix::rowSums(toc),
                                             row.names = rownames(toc)))
  use_to_est <- SoupX::estimateNonExpressingCells(sc, nonExpressedGeneList = soup_gene_sets, clusters = FALSE)
  fallback <- FALSE
  sc <- tryCatch(
    SoupX::calculateContaminationFraction(sc, soup_gene_sets, useToEst = use_to_est, verbose = FALSE),
    error = function(e) {
      message(channel, ": rho could not be estimated (", conditionMessage(e), "); reporting ", rho_fallback)
      fallback <<- TRUE
      SoupX::setContaminationFraction(sc, rho_fallback)
    })
  rho <- sc$metaData$rho[1]
  data.frame(soup_channel            = channel,
             n_cells                 = ncol(toc),
             rho                     = round(rho, 4),
             rho_fallback            = fallback,
             cells_used_for_estimate = sum(rowSums(use_to_est) > 0),
             median_soup_umis_per_cell = round(rho * median(Matrix::colSums(toc)), 1),
             top_soup_genes          = paste(head(rownames(sc$soupProfile)[order(sc$soupProfile$est, decreasing = TRUE)], 5),
                                             collapse = ", "))
}

rna_counts <- LayerData(nsclc_merged, assay = "RNA", layer = "counts")
channels   <- levels(nsclc_merged$soup_channel)
soupx_summary <- do.call(rbind, lapply(channels, function(ch) {
  estimate_soup(rna_counts[, nsclc_merged$soup_channel == ch, drop = FALSE], ch)
}))

# Expected soup UMIs per cell (rho of its channel x library size): what a correction would remove. Counts are unchanged.
soup_umis_expected <- soupx_summary$rho[match(as.character(nsclc_merged$soup_channel), soupx_summary$soup_channel)] *
                      nsclc_merged$nCount_RNA

write.csv(soupx_summary, file.path(table_dir, "nsclc_combined_soupx_rho.csv"), row.names = FALSE)
soupx_summary

In [ ]:
# Contamination per channel, expected soup UMIs per cell, and how often the soup gene sets are detected (uncorrected counts)
p1 <- ggplot(soupx_summary, aes(x = soup_channel, y = rho, fill = rho_fallback)) +
  geom_col() +
  scale_fill_manual(values = c(`FALSE` = "steelblue", `TRUE` = "grey60"), labels = c(`FALSE` = "estimated", `TRUE` = "fallback")) +
  labs(title = "SoupX contamination fraction per channel", x = NULL, y = expression(rho), fill = NULL) +
  theme_bw() + theme(axis.text.x = element_text(angle = 45, hjust = 1))
p2 <- ggplot(data.frame(sample = nsclc_merged$sample, dataset = nsclc_merged$dataset, soup = soup_umis_expected),
             aes(x = sample, y = soup + 1, fill = dataset)) +
  geom_violin(scale = "width") +
  scale_y_log10() +
  labs(title = "Expected soup UMIs per cell (not removed)", x = NULL, y = "rho x nCount_RNA + 1 (log scale)") +
  theme_bw() + theme(axis.text.x = element_text(angle = 45, hjust = 1))

soup_detect <- do.call(rbind, lapply(names(soup_gene_sets), function(s) {
  detected <- Matrix::colSums(rna_counts[soup_gene_sets[[s]], , drop = FALSE]) > 0
  data.frame(set = s, sample = levels(nsclc_merged$sample),
             percent_cells = as.vector(100 * tapply(detected, nsclc_merged$sample, mean)))
}))
soup_detect$sample <- factor(soup_detect$sample, levels = levels(nsclc_merged$sample))
p3 <- ggplot(soup_detect, aes(x = sample, y = percent_cells, fill = set)) +
  geom_col(position = "dodge") +
  labs(title = "Cells detecting each soup gene set (uncorrected counts)", x = NULL, y = "% of cells", fill = NULL) +
  theme_bw() + theme(axis.text.x = element_text(angle = 45, hjust = 1))
rm(rna_counts, soup_detect, soup_umis_expected)

options(repr.plot.width = 18, repr.plot.height = 10)
p <- (p1 | p2) / p3
save_plot(p, "02a_soupx_check.png", width = 18, height = 10)

#### C: Doublets (scDblFinder, per library), before any QC metric

Run **before any QC metric is computed** and before any filter, on the uncorrected counts (Section B only estimated the ambient RNA), on barcodes with ≥ 200 genes. Doublets stay in the object, flagged, until Section F, so the QC plots below can be read with them in mind. Cell Ranger `multi` already removed cross-donor doublets (two CellPlex tags), so scDblFinder simulates doublets **within** each NSCLC_7pt sample. Doublets can only form within one capture, so GSE148071 is run per **library** (barcode prefix), not per patient.

**Finding (P1–P4-only run):** 205 doublets (~3.6% of scored barcodes; 36–72 per sample), mostly with higher UMI and gene counts than singlets. **200 of them pass the MAD thresholds**, so doublet detection removes cells that QC alone would have kept. This rate is above the ~1–1.5% expected for same-donor doublets, which suggests the tag-based demultiplexing did not catch every multiplet.

In [ ]:
mark_stage("C: Doublets (scDblFinder)")

# WSL2: `bash install.sh` installs everything (conda env seurat_env, kernel "R 4.4.3 (seurat_env)").
# Windows R: run once if the packages are missing:
# install.packages("BiocManager"); BiocManager::install(c("scDblFinder", "SingleCellExperiment"))
# Functions are called with pkg:: so the Bioconductor packages are not attached (they would mask some base R names)

# Runs before the QC metrics, on the uncorrected counts (Section B only estimated the ambient RNA). Barcodes with < 200 genes carry too little signal
# to score; they fail the 200-gene floor of the QC filter anyway
dbl_cells <- colnames(nsclc_merged)[nsclc_merged$nFeature_RNA >= 200]
sce <- SingleCellExperiment::SingleCellExperiment(
  assays = list(counts = LayerData(nsclc_merged, assay = "RNA", layer = "counts")[, dbl_cells])
)

# one scDblFinder run per library (doublets form within one capture, so artificial doublets are made within each
# library: GSE148071 = sample + barcode prefix; NSCLC_7pt = sample, since Cell Ranger multi already removed
# cross-donor doublets)
set.seed(seed)
sce <- scDblFinder::scDblFinder(sce, samples = as.character(nsclc_merged@meta.data[dbl_cells, "library"]),
                                BPPARAM = BiocParallel::SerialParam(RNGseed = seed))

dbl_calls <- data.frame(
  scDblFinder.class = factor(as.character(sce$scDblFinder.class), levels = c("singlet", "doublet")),
  scDblFinder.score = sce$scDblFinder.score,
  row.names         = colnames(sce)
)
nsclc_merged <- AddMetaData(nsclc_merged, metadata = dbl_calls)   # barcodes with < 200 genes get NA
rm(sce, dbl_calls)

table(sample = nsclc_merged$sample, scDblFinder.class = nsclc_merged$scDblFinder.class, useNA = "ifany")

In [ ]:
# Doublets should have more UMIs and genes than singlets, in every sample
dbl_meta <- nsclc_merged@meta.data[!is.na(nsclc_merged$scDblFinder.class), ]

dbl_violin <- function(feature) {
  ggplot(dbl_meta, aes(x = scDblFinder.class, y = .data[[feature]], fill = scDblFinder.class)) +
    geom_violin(scale = "width") +
    scale_y_log10() +
    facet_wrap(~ sample, ncol = 8) +
    labs(x = NULL, y = feature) +
    theme_bw() +
    theme(legend.position = "none")
}
p_score <- ggplot(dbl_meta, aes(x = scDblFinder.score, fill = scDblFinder.class)) +
  geom_histogram(bins = 50) +
  facet_wrap(~ sample, ncol = 8) +
  labs(x = "scDblFinder.score", y = "Number of cells", fill = NULL) +
  theme_bw()

options(repr.plot.width = 22, repr.plot.height = 11)
p <- dbl_violin("nCount_RNA") / dbl_violin("nFeature_RNA") / p_score
save_plot(p, "04b_doublets_scDblFinder.png", width = 22, height = 11)

#### D: QC metrics

Same metrics as the tutorial (Section 4), shown per sample, computed on the **uncorrected counts** (no ambient-RNA correction, Section B) after doublet calling. Nothing has been removed yet.

In [ ]:
mark_stage("D: QC metrics")

# Scanpy: sc.pp.calculate_qc_metrics(qc_vars=['mt', 'ribo', 'hb'], percent_top=False, log1p=False)
# nFeature_RNA (= n_genes_by_counts) and nCount_RNA (= total_counts) are added by CreateSeuratObject
nsclc_merged[["percent.mt"]]   <- PercentageFeatureSet(nsclc_merged, pattern = "^MT-")
nsclc_merged[["percent.ribo"]] <- PercentageFeatureSet(nsclc_merged, pattern = "^RP[SL]")   # genes starting with RPL / RPS
nsclc_merged[["percent.hb"]]   <- PercentageFeatureSet(nsclc_merged, pattern = "^HB[ABDEGMQZ][0-9]?$")   # haemoglobins
nsclc_merged$log10GenesPerUMI  <- log10(nsclc_merged$nFeature_RNA) / log10(nsclc_merged$nCount_RNA)   # library complexity

raw_counts <- LayerData(nsclc_merged, assay = "RNA", layer = "counts")
mt_genes   <- grepl("^MT-",    rownames(raw_counts))
ribo_genes <- grepl("^RP[SL]", rownames(raw_counts))
nsclc_merged$total_counts_mt   <- Matrix::colSums(raw_counts[mt_genes, , drop = FALSE])
nsclc_merged$total_counts_ribo <- Matrix::colSums(raw_counts[ribo_genes, , drop = FALSE])

# Ensembl IDs and feature types (Scanpy keeps these in adata.var). Gene expression rows come first in
# features.tsv, so make.unique() of their symbols reproduces the row names Read10X gave the RNA assay
# (Seurat also replaces "_" with "-" in feature names).
gene_info <- feature_tables[[1]][feature_tables[[1]]$feature_types == "Gene Expression", ]
gene_info <- gene_info[match(rownames(raw_counts), gsub("_", "-", make.unique(gene_info$gene_symbol))), ]
stopifnot(!anyNA(gene_info$gene_ids))

# Gene-level QC metrics (Scanpy: adata.var)
gene_qc <- data.frame(
  gene_ids          = gene_info$gene_ids,
  feature_types     = gene_info$feature_types,
  mt                = mt_genes,
  ribo              = ribo_genes,
  n_cells_by_counts = Matrix::rowSums(raw_counts > 0),
  mean_counts       = Matrix::rowMeans(raw_counts),
  total_counts      = Matrix::rowSums(raw_counts),
  row.names         = rownames(raw_counts)
)
gene_qc$pct_dropout_by_counts <- 100 * (1 - gene_qc$n_cells_by_counts / ncol(raw_counts))
nsclc_merged[["RNA"]] <- AddMetaData(nsclc_merged[["RNA"]], metadata = gene_qc)
rm(raw_counts)

head(nsclc_merged@meta.data)

In [ ]:
options(repr.plot.width = 18, repr.plot.height = 5)
qc_features <- c("nFeature_RNA", "nCount_RNA", "percent.mt", "percent.ribo", "percent.hb", "log10GenesPerUMI")

p <- VlnPlot(nsclc_merged, features = qc_features, group.by = "sample", ncol = 6, pt.size = 0.1) & NoLegend()
save_plot(p, "02_qc_violin_raw.png", width = 18, height = 5)

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4)
p <- ggplot(nsclc_merged@meta.data, aes(x = percent.mt, fill = sample)) +
  geom_histogram(bins = 100) +
  labs(title = "Percent mitochondrial counts", x = "percent.mt", y = "Number of cells") +
  theme_classic()
save_plot(p, "03_hist_percent_mt.png", width = 7, height = 4)

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4)
p <- ggplot(nsclc_merged@meta.data, aes(x = percent.ribo, fill = sample)) +
  geom_histogram(bins = 100) +
  labs(title = "Percent ribosomal counts", x = "percent.ribo", y = "Number of cells") +
  theme_classic()
save_plot(p, "04_hist_percent_ribo.png", width = 7, height = 4)

In [ ]:
# UMIs vs detected genes, coloured by mitochondrial fraction, one panel per sample
options(repr.plot.width = 18, repr.plot.height = 9)
p <- ggplot(nsclc_merged@meta.data, aes(x = nCount_RNA, y = nFeature_RNA, colour = percent.mt)) +
  geom_point(size = 0.3, alpha = 0.6) +
  scale_x_log10() + scale_y_log10() +
  scale_colour_viridis_c(option = "magma", direction = -1, limits = c(0, 30), oob = scales::squish) +
  facet_wrap(~ sample, ncol = 4) +
  labs(x = "nCount_RNA (log10)", y = "nFeature_RNA (log10)", colour = "% MT") +
  theme_bw()
save_plot(p, "04b_qc_scatter_by_sample.png", width = 18, height = 9)

#### E: Filtering thresholds (per-sample MADs)

As in the tutorial: outliers beyond 5 MADs on log10 UMIs and log10 genes (both tails), beyond 3 MADs above the median %MT, plus a 200-gene floor. The bounds are computed **per sample**, because depth and composition differ between samples. Nothing is removed until Section F.

**Finding (P1–P4-only run):** in these tumours the 5-MAD bounds on UMIs and genes are extremely wide (lower UMI bound ~9–34, upper bound > 10<sup>5</sup>). Each sample mixes small immune cells with large epithelial cells, so the spread of library sizes is huge. In practice only the **mitochondrial bound (14–17% %MT, depending on the sample)** and the 200-gene floor remove cells.

GSE148071 cells were already filtered by the authors (≥ ~200 genes, ≤ ~5,000 genes), so few of them should fail these bounds; check the table below for how many do.

In [ ]:
mark_stage("E: Filtering thresholds")

mad_bounds <- function(x, nmads) {
  med <- median(x)
  dev <- mad(x)                   # stats::mad() includes the 1.4826 constant
  c(lower = med - nmads * dev, upper = med + nmads * dev)
}
min_genes <- 200

meta <- nsclc_merged@meta.data
qc_bounds <- do.call(rbind, lapply(split(meta, meta$sample), function(m) {
  data.frame(
    sample       = as.character(m$sample[1]),
    counts_lower = 10^mad_bounds(log10(m$nCount_RNA),   5)[["lower"]],
    counts_upper = 10^mad_bounds(log10(m$nCount_RNA),   5)[["upper"]],
    genes_lower  = 10^mad_bounds(log10(m$nFeature_RNA), 5)[["lower"]],
    genes_upper  = 10^mad_bounds(log10(m$nFeature_RNA), 5)[["upper"]],
    mt_upper     =    mad_bounds(m$percent.mt,          3)[["upper"]]
  )
}))
qc_bounds$sample <- factor(qc_bounds$sample, levels = levels(meta$sample))

b <- qc_bounds[match(as.character(meta$sample), as.character(qc_bounds$sample)), ]    # each cell's own sample bounds
nsclc_merged$qc_pass <- meta$nCount_RNA   >= b$counts_lower & meta$nCount_RNA   <= b$counts_upper &
                        meta$nFeature_RNA >= b$genes_lower  & meta$nFeature_RNA <= b$genes_upper  &
                        meta$percent.mt   <= b$mt_upper     & meta$nFeature_RNA >= min_genes
rm(b)

write.csv(qc_bounds, file.path(table_dir, "nsclc_combined_qc_mad_thresholds.csv"), row.names = FALSE)
qc_bounds
table(sample = nsclc_merged$sample, qc_pass = nsclc_merged$qc_pass)

In [ ]:
meta <- nsclc_merged@meta.data
meta$qc_label <- factor(ifelse(meta$qc_pass, "kept", "removed"), levels = c("kept", "removed"))
qc_colours <- c(kept = "grey55", removed = "firebrick")

p1 <- ggplot(meta, aes(x = nCount_RNA, y = nFeature_RNA, colour = qc_label)) +
  geom_point(size = 0.3, alpha = 0.6) +
  geom_vline(data = qc_bounds, aes(xintercept = counts_lower), linetype = "dashed") +
  geom_vline(data = qc_bounds, aes(xintercept = counts_upper), linetype = "dashed") +
  geom_hline(data = qc_bounds, aes(yintercept = genes_lower),  linetype = "dashed") +
  geom_hline(data = qc_bounds, aes(yintercept = genes_upper),  linetype = "dashed") +
  scale_x_log10() + scale_y_log10() + scale_colour_manual(values = qc_colours) +
  facet_wrap(~ sample, ncol = 4) + theme_bw() +
  labs(x = "nCount_RNA (log10)", y = "nFeature_RNA (log10)", colour = NULL)

p2 <- ggplot(meta, aes(x = nCount_RNA, y = percent.mt, colour = qc_label)) +
  geom_point(size = 0.3, alpha = 0.6) +
  geom_hline(data = qc_bounds, aes(yintercept = mt_upper), linetype = "dashed") +
  scale_x_log10() + scale_colour_manual(values = qc_colours) +
  facet_wrap(~ sample, ncol = 4) + theme_bw() +
  labs(x = "nCount_RNA (log10)", y = "percent.mt", colour = NULL)

options(repr.plot.width = 18, repr.plot.height = 16)
p <- (p1 / p2) + plot_layout(guides = "collect")
save_plot(p, "04c_qc_thresholds_by_sample.png", width = 18, height = 16)

#### F: Apply filters (MAD thresholds + singlets)

In [ ]:
mark_stage("F: Apply filters")

# keep = passes the per-sample MAD thresholds AND is a singlet (%in% treats unscored barcodes as not kept)
qc_pass <- nsclc_merged$qc_pass & (nsclc_merged$scDblFinder.class %in% "singlet")

table(qc_pass_MAD = nsclc_merged$qc_pass, scDblFinder.class = nsclc_merged$scDblFinder.class, useNA = "ifany")
table(sample = nsclc_merged$sample, kept = qc_pass)

In [ ]:
nsclc_filtered <- subset(nsclc_merged, cells = colnames(nsclc_merged)[qc_pass])
nsclc_filtered

In [ ]:
# Scanpy: sc.pp.filter_genes(min_cells=3)
min_cells   <- 3
filt_counts <- LayerData(nsclc_filtered, assay = "RNA", layer = "counts")
cells_keep  <- colnames(filt_counts)[Matrix::colSums(filt_counts > 0) >= min_genes]           # cell expresses >= 200 genes
n_cells     <- Matrix::rowSums(filt_counts[, cells_keep, drop = FALSE] > 0)
genes_keep  <- names(n_cells)[n_cells >= min_cells]                                         # gene detected in >= 3 cells
rm(filt_counts)

nsclc_filtered <- subset(nsclc_filtered, cells = cells_keep)

# Filter genes in the RNA assay only. subset(object, features = ...) would apply the gene list to every assay:
# the CMO assay would be dropped and ADT cut down to CD4 / CD14 / CD19 (antibodies named like genes).
# Replacing the assay triggers an expected "Different cells and/or features" warning, which is silenced here.
suppressWarnings(nsclc_filtered[["RNA"]] <- subset(nsclc_filtered[["RNA"]], features = genes_keep))
nsclc_filtered$n_genes <- nsclc_filtered$nFeature_RNA
nsclc_filtered[["RNA"]] <- AddMetaData(nsclc_filtered[["RNA"]],
                                       metadata = data.frame(n_cells = n_cells[genes_keep], row.names = genes_keep))

nsclc_filtered
table(sample = nsclc_filtered$sample)

In [ ]:
options(repr.plot.width = 18, repr.plot.height = 5)
p <- VlnPlot(nsclc_filtered, features = qc_features, group.by = "sample", ncol = 6, pt.size = 0.1) & NoLegend()
save_plot(p, "05_qc_violin_filtered.png", width = 18, height = 5)

#### G: Normalization, HVGs, scaling, PCA

In [ ]:
mark_stage("G: Normalization, HVGs, scaling, PCA")

# Scanpy: sc.pp.normalize_total(target_sum=1e4) + sc.pp.log1p  ->  log1p(counts / total_counts * 1e4)
# Seurat writes the result to the "data" layer and keeps the raw counts in the "counts" layer
nsclc_filtered <- NormalizeData(nsclc_filtered, assay = "RNA", normalization.method = "LogNormalize", scale.factor = 1e4)

# Antibodies: centred log-ratio per cell (standard for CITE-seq / TotalSeq data); used only for plots here
nsclc_filtered <- NormalizeData(nsclc_filtered, assay = "ADT", normalization.method = "CLR", margin = 2)
nsclc_filtered

In [ ]:
n_hvg <- 2000

nsclc_filtered <- FindVariableFeatures(nsclc_filtered, selection.method = "vst", nfeatures = n_hvg)
hvg_genes <- VariableFeatures(nsclc_filtered)        # ordered: most variable gene first

hvg_df <- HVFInfo(nsclc_filtered, method = "vst")
hvg_df$gene_name            <- rownames(hvg_df)
hvg_df$highly_variable      <- hvg_df$gene_name %in% hvg_genes
hvg_df$highly_variable_rank <- match(hvg_df$gene_name, hvg_genes)
hvg_df <- hvg_df[hvg_df$highly_variable, ]          # keep only the HVGs (genome order)

write.csv(hvg_df, file.path(table_dir, "nsclc_combined_hvg.csv"), row.names = FALSE)
head(hvg_df)

In [ ]:
top10_hvgs <- hvg_df$gene_name[order(hvg_df$highly_variable_rank)][1:10]
top10_hvgs

In [ ]:
# Seurat's standard HVG plot: mean vs standardised variance, top 10 labelled
options(repr.plot.width = 8, repr.plot.height = 5)
p <- LabelPoints(plot = VariableFeaturePlot(nsclc_filtered), points = top10_hvgs, repel = TRUE)
save_plot(p, "06_variable_features.png", width = 8, height = 5)

In [ ]:
# Violin plot of the top 10 HVGs in one panel (Scanpy: sc.pl.violin(..., multi_panel=False))
expr_mat  <- as.matrix(LayerData(nsclc_filtered, assay = "RNA", layer = "data")[top10_hvgs, ])
expr_long <- data.frame(
  gene       = factor(rep(top10_hvgs, times = ncol(expr_mat)), levels = top10_hvgs),
  expression = as.vector(expr_mat)
)
rm(expr_mat)

options(repr.plot.width = 10, repr.plot.height = 5)
p <- ggplot(expr_long, aes(x = gene, y = expression, fill = gene)) +
  geom_violin(scale = "width", trim = TRUE) +
  geom_jitter(width = 0.2, size = 0.1, alpha = 0.3) +
  labs(x = "Genes", y = "Expression (normalised log1p values)", title = "Violin Plot of Top 10 HVGs") +
  theme_classic() +
  theme(axis.text.x  = element_text(angle = 45, hjust = 1),
        axis.title   = element_text(size = 12),
        plot.title   = element_text(size = 14),
        legend.position = "none")
save_plot(p, "07_violin_top10_hvgs.png", width = 10, height = 5)

In [ ]:
# Scanpy: sc.pp.scale(max_value=10); sc.pp.pca
# Each HVG is z-scored (unit variance, clipped at 10) as in the tutorial, so every gene has equal weight in the PCA
nsclc_filtered <- ScaleData(nsclc_filtered, features = hvg_genes)
nsclc_filtered <- RunPCA(nsclc_filtered, features = hvg_genes, npcs = 50, verbose = FALSE)

In [ ]:
# PCA coloured by dataset and by sample, plus two compartment markers: PTPRC (CD45, immune) and EPCAM (epithelial / tumour)
var_ratio <- pca_variance_ratio(nsclc_filtered)
pc_labs   <- labs(x = sprintf("PC1 (%.1f%%)", 100 * var_ratio$variance_ratio[1]),
                  y = sprintf("PC2 (%.1f%%)", 100 * var_ratio$variance_ratio[2]))
pca_genes <- intersect(c("PTPRC", "EPCAM"), rownames(nsclc_filtered))

pca_plots <- c(list(DimPlot(nsclc_filtered, reduction = "pca", group.by = "dataset", shuffle = TRUE, seed = seed) + pc_labs,
                    DimPlot(nsclc_filtered, reduction = "pca", group.by = "sample",  shuffle = TRUE, seed = seed) + pc_labs),
               lapply(pca_genes, function(g) FeaturePlot(nsclc_filtered, features = g, reduction = "pca") + pc_labs))

options(repr.plot.width = 6 * length(pca_plots), repr.plot.height = 5)
p <- wrap_plots(pca_plots, nrow = 1)
save_plot(p, "08_pca_sample_markers.png", width = 6 * length(pca_plots), height = 5)

In [ ]:
# Scanpy: sc.pl.pca_variance_ratio(n_pcs=50); ElbowPlot is the usual Seurat equivalent (standard deviation per PC)
options(repr.plot.width = 12, repr.plot.height = 4.5)
p <- ggplot(var_ratio, aes(x = PC, y = variance_ratio)) +
  geom_point() +
  labs(title = "PCA variance ratio", x = "Principal component", y = "Variance ratio") +
  theme_classic()
p <- p | ElbowPlot(nsclc_filtered, ndims = 50)
save_plot(p, "09_pca_variance_ratio.png", width = 12, height = 4.5)

In [ ]:
# Scanpy: sc.pl.pca_loadings(components=(1, 2), include_lowest=True)
# balanced = TRUE shows the most positive and the most negative loadings
options(repr.plot.width = 10, repr.plot.height = 6)
p <- VizDimLoadings(nsclc_filtered, dims = 1:2, reduction = "pca", nfeatures = 30, balanced = TRUE)
save_plot(p, "10_pca_loadings_PC1_PC2.png", width = 10, height = 6)

##### Number of PCs

Stage 1 read the elbow off the PBMC plot (7 PCs). The NSCLC data have their own elbow, so the cell below picks the number of PCs with a common rule of thumb (from the HBC single-cell training). It uses each PC's share of the total standard deviation, and takes whichever of these two PCs comes first:

- the PC where the cumulative share passes 90% while the PC on its own adds less than 5%
- the PC just after the last drop of more than 0.1 percentage points between consecutive PCs

Compare the result with the elbow plot above, and set `n_pcs` by hand if you disagree.

In [ ]:
stdev <- Stdev(nsclc_filtered, reduction = "pca")
pct   <- 100 * stdev / sum(stdev)
cumu  <- cumsum(pct)

pc_cumulative <- which(cumu > 90 & pct < 5)[1]
pc_drop       <- tail(which(diff(pct) < -0.1), 1) + 1
n_pcs <- min(pc_cumulative, pc_drop, na.rm = TRUE)
# n_pcs <- 10   # uncomment to set it by hand after looking at the elbow plot

nsclc_filtered@misc$n_pcs <- n_pcs   # stored in the object for the Harmony notebook
c(pc_cumulative = pc_cumulative, pc_drop = pc_drop, n_pcs = n_pcs)

#### H: Is the structure driven by batch (dataset, sample) or by biology?

**What a batch effect does.** When samples are processed separately (different lab, day, dissociation, platform, chemistry or sequencing depth), every cell of a batch receives a shared technical shift. Without integration this causes four problems:

* the same cell type splits into **batch-specific clusters**, inflating the number of "cell types";
* marker genes and cell-type labels become **confounded with the batch**;
* differential expression between conditions processed in different batches cannot separate biology from technique;
* integration methods are then needed, with the risk of **over-correcting** real biological differences.

**Two contrasts in one dataset.**

| Batch label | What differs | Expectation |
|---|---|---|
| `sample` within NSCLC_7pt (P1–P4) | donors only: one CellPlex capture | **negative control**, no technical batch effect (shown in the P1–P4-only run) |
| `dataset` (NSCLC_7pt vs GSE148071) | lab, tissue handling, platform, chemistry, gene annotation, depth, cell filtering | **technical batch effect expected** |

**The test.** Cells of the **same type from different batches** should overlap if there is no batch effect. All tests use the PCA only. Biology is represented by **coarse marker-based compartments** (module scores, not clustering). Malignant **epithelial** cells are genuinely patient-specific, and the cell-type composition differs between studies (advanced-stage biopsies in GSE148071), so the fair test is on the **shared immune and stromal compartments**: the `..._given_compartment` R² and the silhouette within the largest shared compartment.

In [ ]:
mark_stage("H: Batch vs biology diagnostics")

# Coarse compartment label per cell = highest module score among marker sets (no graph, no clustering)
compartment_markers <- list(
  T_NK        = c("CD3D", "CD3E", "CD2", "NKG7", "GNLY"),
  B_plasma    = c("MS4A1", "CD79A", "JCHAIN", "MZB1"),
  Myeloid     = c("CD68", "CD14", "LYZ", "C1QA", "CD163"),
  Epithelial  = c("EPCAM", "KRT19", "KRT8", "KRT18"),
  Fibroblast  = c("COL1A1", "COL1A2", "DCN", "LUM"),
  Endothelial = c("PECAM1", "VWF", "CLDN5")
)
compartment_markers <- lapply(compartment_markers, intersect, rownames(nsclc_filtered))
compartment_markers <- compartment_markers[lengths(compartment_markers) > 0]

nsclc_filtered <- AddModuleScore(nsclc_filtered, features = compartment_markers, name = "cmp_score", seed = seed)
score_cols <- paste0("cmp_score", seq_along(compartment_markers))
scores <- as.matrix(nsclc_filtered@meta.data[, score_cols])
colnames(scores) <- names(compartment_markers)
best <- colnames(scores)[max.col(scores, ties.method = "first")]
nsclc_filtered$compartment <- factor(ifelse(apply(scores, 1, max) > 0.1, best, "unassigned"),   # 0.1: weak scores stay unassigned
                                     levels = c(names(compartment_markers), "unassigned"))
nsclc_filtered@meta.data[, score_cols] <- NULL
rm(scores, best)

table(compartment = nsclc_filtered$compartment, sample = nsclc_filtered$sample)

In [ ]:
# The same PCA coloured by dataset (left), sample (middle) and compartment (right)
pc_plot <- function(group, dims) {
  DimPlot(nsclc_filtered, reduction = "pca", dims = dims, group.by = group, shuffle = TRUE, seed = seed, pt.size = 0.3) +
    ggtitle(paste0(group, ": PC", dims[1], " vs PC", dims[2]))
}
options(repr.plot.width = 21, repr.plot.height = 11)
p <- (pc_plot("dataset", c(1, 2)) | pc_plot("sample", c(1, 2)) | pc_plot("compartment", c(1, 2))) /
     (pc_plot("dataset", c(3, 4)) | pc_plot("sample", c(3, 4)) | pc_plot("compartment", c(3, 4)))
save_plot(p, "16_pca_dataset_sample_vs_compartment.png", width = 21, height = 11)

In [ ]:
# How much of each PC's variance does the batch label explain, compared with the cell compartment?
#   dataset, sample, compartment: R^2 of PC ~ label, all cells
#   ... | compartment: extra R^2 from adding the batch label to PC ~ compartment, shared immune / stromal cells only
pcs  <- Embeddings(nsclc_filtered, reduction = "pca")[, 1:n_pcs]
meta <- nsclc_filtered@meta.data
r2     <- function(y, g)      summary(lm(y ~ g))$r.squared          # R^2 of one label
r2_two <- function(y, g1, g2) summary(lm(y ~ g1 + g2))$r.squared     # R^2 of two labels together

shared <- meta$compartment %in% c("T_NK", "B_plasma", "Myeloid", "Fibroblast", "Endothelial")
cmp_s  <- droplevels(meta$compartment[shared])
dst_s  <- droplevels(meta$dataset[shared])
smp_s  <- droplevels(meta$sample[shared])

r2_pcs <- data.frame(
  PC                        = seq_len(n_pcs),
  variance_share            = Stdev(nsclc_filtered, reduction = "pca")[1:n_pcs]^2,
  dataset                   = apply(pcs, 2, r2, meta$dataset),
  sample                    = apply(pcs, 2, r2, meta$sample),
  compartment               = apply(pcs, 2, r2, meta$compartment),
  dataset_given_compartment = apply(pcs[shared, ], 2, function(y) r2_two(y, cmp_s, dst_s) - r2(y, cmp_s)),
  sample_given_compartment  = apply(pcs[shared, ], 2, function(y) r2_two(y, cmp_s, smp_s) - r2(y, cmp_s))
)
r2_pcs$variance_share <- r2_pcs$variance_share / sum(r2_pcs$variance_share)
write.csv(r2_pcs, file.path(table_dir, "nsclc_combined_preInt_pc_r2_batch_vs_compartment.csv"), row.names = FALSE)

# Summary over the first n_pcs PCs, weighted by each PC's share of variance
r2_cols <- c("dataset", "sample", "compartment", "dataset_given_compartment", "sample_given_compartment")
round(sapply(r2_pcs[, r2_cols], weighted.mean, w = r2_pcs$variance_share), 3)

In [ ]:
r2_long <- tidyr::pivot_longer(r2_pcs, tidyr::all_of(r2_cols), names_to = "explained_by", values_to = "R2")
r2_long$explained_by <- factor(r2_long$explained_by, levels = r2_cols,
                               labels = c("dataset (all cells)", "sample (all cells)", "compartment (all cells)",
                                          "dataset within shared compartments", "sample within shared compartments"))

options(repr.plot.width = 14, repr.plot.height = 4.5)
p <- ggplot(r2_long, aes(x = factor(PC), y = R2, fill = explained_by)) +
  geom_col(position = "dodge") +
  scale_fill_manual(values = c("purple4", "firebrick", "steelblue", "orchid3", "darkorange2")) +
  labs(title = "Variance of each PC explained by batch (dataset, sample) vs biology (compartment)",
       x = "Principal component", y = expression(R^2), fill = NULL) +
  theme_bw() + theme(legend.position = "top")
save_plot(p, "17_pc_r2_batch_vs_compartment.png", width = 14, height = 4.5)

In [ ]:
# Within one shared cell compartment (same cell type in every patient), do the datasets / samples still separate?
largest_shared <- names(which.max(table(cmp_s)))
cells_cmp <- colnames(nsclc_filtered)[meta$compartment == largest_shared]

cmp_plot <- function(group) {
  DimPlot(nsclc_filtered, reduction = "pca", cells = cells_cmp, group.by = group, shuffle = TRUE, seed = seed, pt.size = 0.4) +
    ggtitle(paste0(largest_shared, " cells only: PC1 vs PC2, by ", group))
}
options(repr.plot.width = 14, repr.plot.height = 5.5)
p <- cmp_plot("dataset") | cmp_plot("sample")
save_plot(p, "18_pca_within_compartment_by_dataset_sample.png", width = 14, height = 5.5)

In [ ]:
# Average silhouette width on PCA distances (3,000 random cells): how well does a label separate cells?
# > 0: cells are closer to their own group than to others; ~ 0: groups overlap. Labels are given, not clustered.
set.seed(seed)
avg_silhouette <- function(labels, emb) {
  summary(cluster::silhouette(as.integer(factor(labels)), dist(emb)))$avg.width
}
idx_all    <- sample(nrow(pcs), min(3000, nrow(pcs)))
idx_shared <- sample(which(meta$compartment == largest_shared), min(3000, sum(meta$compartment == largest_shared)))

silhouettes <- c(
  dataset_all_cells             = avg_silhouette(meta$dataset[idx_all],     pcs[idx_all, ]),
  sample_all_cells              = avg_silhouette(meta$sample[idx_all],      pcs[idx_all, ]),
  compartment_all_cells         = avg_silhouette(meta$compartment[idx_all], pcs[idx_all, ]),
  dataset_within_largest_shared = avg_silhouette(meta$dataset[idx_shared],  pcs[idx_shared, ]),
  sample_within_largest_shared  = avg_silhouette(meta$sample[idx_shared],   pcs[idx_shared, ])
)
round(silhouettes, 3)

**Results.** The numbers below are from the run **with NSCLC_7pt P1–P4 only**. Re-running the notebook adds the `dataset` rows to the tables and figures above; update this cell with them.

| Measure (first 14 PCs), P1–P4 only | Sample | Compartment |
|---|---|---|
| Variance explained, weighted by PC variance (R²) | 0.09 | **0.39** |
| Extra variance explained by sample *within* shared immune / stromal compartments | **0.01** | n/a |
| Average silhouette width, all cells | −0.02 | **0.23** |
| Average silhouette width by sample, T/NK cells only | −0.02 | n/a |

* **Biology dominated the leading PCs** (compartment explained 66% of PC1, 38% of PC2, 54% of PC3, 77% of PC4).
* **Shared cell types mixed across P1–P4** (T/NK cells overlapped completely): the signature of **no technical batch effect**.
* **Where sample mattered, it was biology:** patient-specific tumour cells (P2's and P1's epithelial / unassigned arms) and composition (P1, P2 epithelium-rich; P4 B/plasma-rich).

**How to read the `dataset` results.**

| Diagnostic | No technical batch effect | Technical batch effect |
|---|---|---|
| `dataset_given_compartment` R² (weighted) | ≈ 0, like `sample_given_compartment` in P1–P4 (0.01) | clearly > 0, often concentrated in one or two leading PCs |
| Silhouette by dataset within the largest shared compartment | ≈ 0 | > 0 |
| PCA of T/NK (or myeloid) cells coloured by dataset | one overlapping cloud | two separate clouds |
| `sample_given_compartment` vs `dataset_given_compartment` | both ≈ 0 | sample ≈ dataset: the sample signal is mostly the dataset |

**Impact for the next step.**

1. If `dataset` separates shared cell types, clustering this uncorrected PCA would split T/NK, myeloid and B cells into NSCLC_7pt and GSE148071 clusters: too many "cell types", and markers confounded with the study. **Integration is then required**, with `dataset` (or `library`) as the batch key in Harmony.
2. After integration, re-run these diagnostics: `dataset_given_compartment` should fall to ≈ 0, the compartment silhouette should stay high, and tumour cells should remain patient-specific. If tumour cells from different patients are forced together, the correction is too strong.
3. P1–P4 remain the negative control: they mixed without any correction, so a method that changes them a lot is over-correcting.
4. `dataset` is not purely technical: GSE148071 holds different patients and advanced-stage biopsies, so differences in composition and in tumour cells are biology. Only the separation **within** shared compartments counts as batch effect.

#### I: Save

In [ ]:
mark_stage("I: Save")

saveRDS(nsclc_filtered, file = rds_file)
write.csv(nsclc_filtered@meta.data, file.path(table_dir, "nsclc_combined_preInt_cell_metadata.csv"))

`data/nsclc_combined_preInt.rds` holds the shared genes (`counts`, not corrected for ambient RNA, and `data`), scaled HVGs, the PCA, the `dataset`, `sample`, `library`, `soup_channel` and `compartment` labels and `@misc$n_pcs` (SoupX ρ per channel: `nsclc_combined_soupx_rho.csv`). The ADT and CMO assays cover the NSCLC_7pt cells only. It has no neighbour graph, clusters or UMAP.

**Summary of the P1–P4-only run:** 5,751 → 4,720 cells (P1 1,362, P2 1,518, P3 895, P4 945) and 25,926 genes after QC and doublet removal; the structure followed **cell type, not sample**, because CellPlex multiplexing prevented a technical batch effect. Adding GSE148071 (19,951 author-filtered cells, 23,819 shared genes) tests the opposite case: a separately processed study on another platform, where a **dataset-driven batch effect** in shared cell types is expected and would make integration necessary.

#### Runtime per stage

How long each stage took in this run, from the `mark_stage()` calls at the start of every stage (defined in the setup cell). The table is also written to `data/tables/NSCLC_combined/nsclc_combined_runtime_per_stage.csv`.

* **Elapsed** = wall-clock time from the start of one stage to the start of the next. It is only meaningful when the notebook runs top to bottom without pauses (Run All, or `jupyter nbconvert --execute`); when cells are run by hand it also counts the time spent reading.
* **CPU** = processor time used by R in that stage. It ignores idle time, and can exceed the elapsed time if the BLAS library uses several threads.

In [ ]:
mark_stage("End")
marks <- data.frame(stage = names(stage_marks),
                    time  = vapply(stage_marks, `[[`, numeric(1), "time"),
                    cpu   = vapply(stage_marks, `[[`, numeric(1), "cpu"),
                    row.names = NULL)
marks <- marks[order(marks$time), ]          # a stage that was re-run counts from its latest start

runtimes <- data.frame(
  stage           = head(marks$stage, -1),
  elapsed_seconds = round(diff(marks$time), 1),
  cpu_seconds     = round(diff(marks$cpu), 1)
)
runtimes$elapsed_minutes  <- round(runtimes$elapsed_seconds / 60, 2)
runtimes$percent_of_total <- round(100 * runtimes$elapsed_seconds / sum(runtimes$elapsed_seconds), 1)
write.csv(runtimes, file.path(table_dir, "nsclc_combined_runtime_per_stage.csv"), row.names = FALSE)

rbind(runtimes,
      data.frame(stage = "Total", elapsed_seconds = sum(runtimes$elapsed_seconds), cpu_seconds = sum(runtimes$cpu_seconds),
                 elapsed_minutes = round(sum(runtimes$elapsed_seconds) / 60, 2), percent_of_total = 100))

runtimes$stage <- factor(runtimes$stage, levels = rev(runtimes$stage))   # first stage at the top
options(repr.plot.width = 9, repr.plot.height = 1.5 + 0.4 * nrow(runtimes))
p <- ggplot(runtimes, aes(x = elapsed_seconds, y = stage)) +
  geom_col(fill = "steelblue") +
  geom_text(aes(label = sprintf("%.1f s (%.0f%%)", elapsed_seconds, percent_of_total)), hjust = -0.1, size = 3.5) +
  scale_x_continuous(expand = expansion(mult = c(0, 0.25))) +
  labs(title = "Runtime per stage (wall-clock)", x = "Elapsed time (s)", y = NULL) +
  theme_bw()
save_plot(p, "19_runtime_per_stage.png", width = 9, height = 1.5 + 0.4 * nrow(runtimes))

In [ ]:
sessionInfo()